# 01 — Prepare & Merge Training Data

**Primary**: MetaBreath demo dataset (1,200 rows — NSC 2026 official data)
**Secondary**: eNose TGS sensor dataset (1,000 rows — NSC-recommended Kaggle)
**Tertiary**: UCI Gas Drift Acetone subset (for drift analysis)

Labels from MetaBreath demo: **low / moderate / high / unreliable**
- low: acetone_delta < 30 ppm
- moderate: 30–74.9 ppm
- high: ≥ 75 ppm

**Outputs**: `data/processed/merged_training.csv`, `data/processed/X_train.csv`, etc.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path('../../..')
NSC_DATA = ROOT / 'แข่งชนะ by Coach Bright_NSC' / 'NSC 2026' / '1. Data set สำหรับเทรน AI'
KAGGLE   = ROOT / 'data' / 'kaggle'
PROCESSED = ROOT / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)

print('Paths OK:', NSC_DATA.exists(), KAGGLE.exists())

## 1. Load MetaBreath Demo Dataset (Primary — NSC 2026)

In [ ]:
demo_path = NSC_DATA / 'metabreath_acetone_delta_demo_dataset.csv'
demo = pd.read_csv(demo_path)
print('MetaBreath demo shape:', demo.shape)
print('Columns:', demo.columns.tolist())
print('Label distribution:')
print(demo['label'].value_counts())
demo.head(3)

In [ ]:
# Label boundaries (confirmed from actual data)
print('acetone_delta by label:')
for lbl in ['low', 'moderate', 'high']:
    subset = demo[demo['label'] == lbl]['acetone_delta']
    print(f'  {lbl}: {subset.min():.1f} – {subset.max():.1f} ppm (mean={subset.mean():.1f})')

In [ ]:
# Encode labels as integers for sklearn
LABEL_MAP = {'low': 0, 'moderate': 1, 'high': 2, 'unreliable': -1}
demo['label_int'] = demo['label'].map(LABEL_MAP)

# Drop unreliable samples from training
demo_clean = demo[demo['label_int'] >= 0].copy()
print(f'After removing unreliable: {len(demo_clean)} rows')
print(demo_clean['label'].value_counts())

## 2. MetaBreath Feature Matrix

All 13 AI features defined in README_dataset_sources_and_usage.md

In [ ]:
FEATURE_COLS = [
    'acetone_delta', 'ambient_voc',
    'pressure_mean', 'pressure_std', 'breath_duration',
    'temperature', 'humidity',
    'quality_score', 'reliability_score',
    'ketosis_index', 'metabolic_score', 'fat_burning_index', 'lifestyle_score',
]
TARGET_COL = 'label_int'

X_demo = demo_clean[FEATURE_COLS]
y_demo = demo_clean[TARGET_COL]

print('Feature shape:', X_demo.shape)
print('Missing values:', X_demo.isnull().sum().sum())
X_demo.describe()

## 3. Load eNose Dataset (Augmentation — TGS sensor alignment)

In [ ]:
enose_path = KAGGLE / 'enose-diseases' / 'enose_dataset_to_predict_human_disease.csv'
enose = pd.read_csv(enose_path)
print('eNose shape:', enose.shape)
print('Subjek distribution:', enose['Subjek'].value_counts().to_dict())

In [ ]:
# Align eNose to MetaBreath feature space
# TGS2620 (alcohol/VOC) → breath_voc proxy (closest to TGS1820)
# Mean of air sensors → ambient_voc proxy
enose['ambient_voc']    = enose[['TGS2600', 'TGS2602']].mean(axis=1)
enose['acetone_delta']  = (enose['TGS2620'] - enose['ambient_voc']).clip(lower=0)

# Map binary label (Diabetes=high, Normal=low)
# Diabetes samples are treated as 'high' metabolic state for training
enose['label_int'] = enose['Subjek'].map({'Diabetes': 2, 'Normal': 0})

# Synthetic feature fillers (mean from MetaBreath demo)
for col in FEATURE_COLS:
    if col not in enose.columns:
        enose[col] = X_demo[col].mean()

# Override the sensor-derived columns
enose['ambient_voc']   = enose['ambient_voc']
enose['acetone_delta'] = enose['acetone_delta']

X_enose = enose[FEATURE_COLS].copy()
y_enose = enose['label_int'].copy()

print('eNose aligned shape:', X_enose.shape)

## 4. Merge + Train/Test Split

In [ ]:
import pandas as pd

X_all = pd.concat([X_demo, X_enose], ignore_index=True).fillna(0)
y_all = pd.concat([y_demo, y_enose], ignore_index=True)

print(f'Merged dataset: {len(X_all)} rows')
print('Label distribution:', dict(pd.Series(y_all).value_counts()))
print('Label mapping: 0=low, 1=moderate, 2=high')

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

X_train.to_csv(PROCESSED / 'X_train.csv', index=False)
X_test.to_csv(PROCESSED  / 'X_test.csv',  index=False)
y_train.to_csv(PROCESSED / 'y_train.csv', index=False)
y_test.to_csv(PROCESSED  / 'y_test.csv',  index=False)

X_all.to_csv(PROCESSED / 'merged_training.csv', index=False)
print('Saved all files to data/processed/')

## 5. Load UCI Gas Drift — Acetone (Drift Reference Only)

In [ ]:
from sklearn.datasets import load_svmlight_file

drift_dir = KAGGLE / 'uci-gas-drift' / 'Dataset'
all_batches = []

for i in range(1, 11):
    path = drift_dir / f'batch{i}.dat'
    if not path.exists():
        continue
    X_b, y_b = load_svmlight_file(str(path))
    df_b = pd.DataFrame(X_b.toarray(), columns=[f'feat_{j}' for j in range(X_b.shape[1])])
    df_b['gas_type'] = y_b.astype(int)
    df_b['batch'] = i
    all_batches.append(df_b)

drift_df = pd.concat(all_batches, ignore_index=True)
acetone_df = drift_df[drift_df['gas_type'] == 5].copy()
print('UCI Acetone rows by batch:')
print(acetone_df['batch'].value_counts().sort_index())

acetone_df.to_csv(PROCESSED / 'uci_acetone_only.csv', index=False)
print('Saved uci_acetone_only.csv — use for drift analysis in notebook 03')

## 6. Save MetaBreath NPY (for LSTM in notebook 04)

In [ ]:
import zipfile

npy_zip = NSC_DATA / 'metabreath_features.zip'
with zipfile.ZipFile(npy_zip) as zf:
    zf.extractall(PROCESSED)
    print('Extracted:', zf.namelist())

features_npy = PROCESSED / 'metabreath_features.npy'
if features_npy.exists():
    arr = np.load(features_npy, allow_pickle=True)
    print('metabreath_features.npy shape:', arr.shape if hasattr(arr, 'shape') else type(arr))

In [ ]:
print('\n=== Data preparation complete ===')
print(f'MetaBreath demo (primary):  {len(X_demo)} rows')
print(f'eNose (augment):            {len(X_enose)} rows')
print(f'Total merged:               {len(X_all)} rows')
print(f'Train split:                {len(X_train)} rows')
print(f'Test split:                 {len(X_test)} rows')
print(f'UCI Acetone (drift ref):    {len(acetone_df)} rows')
print('\nNext: Run 02_random_forest.ipynb')